# Stronger speech experiments: fresh evaluation

Reproduce the v2 ASR, speaker verification and multilingual intent experiments. The experiment protocol is in `experiments/v2_protocol.json`. Earlier holdouts are retired; all v2 test speakers/intent utterances are excluded from earlier evaluations as specified in the data builders.

The notebook cells are unexecuted; the same CLI programs were run in the project environment. Measured artifacts are under `evidence/2026-11-19-v2/`. Source transcripts and recording identifiers are downloaded locally and are not republished in that bundle. Models and data require several GB of disk. This tested CPU recipe uses no paid APIs; runtime depends on available CPU resources.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
repo = Path('/content/speech-v2') if Path('/content').exists() else Path.cwd() / 'speech-v2'
if repo.exists(): raise RuntimeError('Choose a fresh checkout directory')
subprocess.run(['git','clone','--branch','codex/siri-real-speech-lab-20261119','https://github.com/ByteBoss123/Speech-Command-.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','torch==2.8.0','--index-url','https://download.pytorch.org/whl/cpu'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-speech.txt'],check=True)
def run(module, *arguments):
    subprocess.run([sys.executable,'-m',module,*arguments],check=True)
print('Code commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

## Reconstruct exclusions and prepare fresh data

Recreate the original manifests to identify what earlier experiments already used. Output directories must be empty. The expanded ASR data use 48 training speakers and 20 new evaluation speakers in each of validation/test. Enrollment and probe recordings come from different chapters. The intent test excludes every previous source recording, normalized transcript and audio checksum; old test examples may enter new training.

In [ ]:
run('speechlab.prepare','--revision','71cacbfb7e2354c4226d01e70d77d5fca3d04ba1')
run('speechlab.intent_data')
run('speechlab.prepare_v2')
run('speechlab.intent_data_v2')

## ASR: normalized targets, conservative adaptation, generated-WER selection

Train decoder LoRA with a lower learning rate, warmup/decay and modest training noise. Select among the unchanged baseline and three epochs using generated validation WER. Test is scored only after selection is saved.

The larger pretrained Whisper-base checkpoint is also evaluated. A fixed validation-WER rule chooses model capacity; selecting a better pretrained model does not establish a fine-tuning gain.

In [ ]:
run('speechlab.asr_v2')
run('speechlab.asr_capacity')
run('speechlab.select_asr')

## Speaker verification: different chapters and multi-recording enrollment

Compare a single enrollment recording, three-recording enrollment and training-cohort centering. Calibrate thresholds at a 1% empirical validation FAR target, choose the method on validation, then freeze it. Cross-chapter is harder than the earlier same-chapter trials, but does not guarantee different sessions or microphones.

In [ ]:
run('speechlab.speaker_v2')

## Intent: expanded data, three seeds, validation-selected fusion

Both encoders stay frozen. Fit 3 seeds for each audio/text/fusion head, then select a blend of text and fused probabilities using validation only. A zero fusion weight means the simpler text ensemble was selected. Do not select a different model after reading test scores.

In [ ]:
run('speechlab.intent_features','--manifest','data/minds14-v2/manifest.jsonl','--output','runs/intent-features-v2')
run('speechlab.intent_v2')

In [ ]:
report=json.loads(Path('runs/intent-v2/metrics.json').read_text())
for model, result in report['models'].items():
    test=result['test']
    print(model, 'accuracy:',round(test['accuracy'],4), 'macro F1:',round(test['macro_f1'],4))
print('Selected fusion weight:',report['selected_audio_text_weight'])
print('Compared with frozen v1:',report['selected_minus_previous'])

## Recompute committed evidence and run tests

The verifier recomputes error rates, confidence intervals, validation choices and intent metrics from saved public evidence. For an independently regenerated run, compare its metrics with the committed scorecards; small hardware-related differences may occur.

In [ ]:
run('pytest','tests_speech','-q')
run('speechlab.verify_v2')

## Run exported intent inference

The first new test example is used without correctness-based selection. Supply your own corresponding recording/transcript to try a different example. This is a closed-set banking classifier, not an assistant-directed-speech detector.

In [ ]:
rows=[json.loads(x) for x in Path('data/minds14-v2/manifest.jsonl').read_text().splitlines()]
example=next(r for r in rows if r['split']=='test')
print('Reference:',example['intent'])
run('speechlab.intent_ensemble_predict',str(Path('data/minds14-v2')/example['path']),'--transcript',example['text'],'--model','runs/intent-v2')

All results remain research pilots. MInDS speaker IDs are unavailable, foundation-model overlap is unknown, and synthetic noise is not a realistic environmental-noise benchmark. Repeated seed measurements do not establish population-level uncertainty. Implementation was completed with AI assistance.